In [1]:
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup

In [2]:
headers = {
    "User-Agent":"Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:150.0) Gecko/20100101 Firefox/150.0"
}

In [3]:
n_season = 2025
n_match = 0

all_rounds = []

for n_round in range(1,39):
    print(f'{n_round}, ', end="")
    # Scraping from url
    url = f"https://www.transfermarkt.com.br/premier-league/spieltag/wettbewerb/GB1/saison_id/{n_season}/spieltag/{n_round}"
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    all_information = soup.find_all('table', {'style':'border-top: 0 !important;'})

    # Gathering Useful Information
    for row in all_information:
        temp = []
        n_match += 1

        season_key = f'PL-{n_season}'
        match_key = f'M-{n_season}-{n_match:03d}'

        if n_round < 10: round_key = 'R-2025-0' + str(n_round)
        else: round_key = 'R-2025-' + str(n_round)

        temp.append(season_key)
        temp.append(round_key)
        temp.append(match_key)

        # List with the entire class necesaire to get the home and away team's names
        gross_home_team = row.find('td', {'class':'rechts hauptlink no-border-rechts hide-for-small spieltagsansicht-vereinsname'})
        gross_away_team = row.find('td', {'class':'hauptlink zentriert no-border-rechts no-border-links hide-for-small spieltagsansicht-wappen'})

        # Checking for a possible forum buttom
        home_forum_check = gross_home_team.find('a').get('href')
        away_forum_check = gross_away_team.find('a').get('href')

        # Different ways to get the title depending if it has the forum buttom
        if 'forum' in home_forum_check and 'forum' in away_forum_check:
            home_team = gross_home_team.find_all('a')[1].get('title')
            away_team = gross_away_team.find_all('a')[1].get('title')
        elif 'forum' in home_forum_check:
            home_team = gross_home_team.find_all('a')[1].get('title')
            away_team = gross_away_team.find('a').get('title')
        elif 'forum' in away_forum_check:
            home_team = gross_home_team.find('a').get('title')
            away_team = gross_away_team.find_all('a')[1].get('title')
        else:
            home_team = gross_home_team.find('a').get('title')
            away_team = gross_away_team.find('a').get('title')

        # Getting the final score
        final_score = row.find('span', {'class':'matchresult finished'}).string

        # Appending data from a single match together
        temp.append(home_team)
        temp.append(final_score)
        temp.append(away_team)

        # Storing adicional info separately, easier to extract right information
        adicional_info = row.find_all('td', {'class':'zentriert no-border'})

        for i, item in enumerate(adicional_info):
            if i == 2:
                text = item.get_text(" ", strip=True)
                try: 
                    attendance = text.split()[0]
                    temp.append(attendance)
                except: temp.append(text)
            else:
                day_ref = item.find('a').string
                temp.append(day_ref.strip())

        all_rounds.append(temp)

df_all_rounds = pd.DataFrame(all_rounds)
df_all_rounds.columns = ['season_id','round_id', 'match_id', 'home_team', 'final_score', 'away_team', 'date', 'referee', 'attendance']
display(df_all_rounds)


1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 

,season_id,round_id,match_id,home_team,final_score,away_team,date,referee,attendance
0,PL-2025,R-2025-01,M-2025-001,FC Liverpool,4:2,AFC Bournemouth,15/08/2025,Anthony Taylor,60.315
1,PL-2025,R-2025-01,M-2025-002,Aston Villa FC,0:0,Newcastle United,16/08/2025,Craig Pawson,42.526
2,PL-2025,R-2025-01,M-2025-003,Brighton & Hove Albion,1:1,FC Fulham,16/08/2025,Samuel Barrott,31.478
3,PL-2025,R-2025-01,M-2025-004,AFC Sunderland,3:0,West Ham United,16/08/2025,Robert Jones,46.233
4,PL-2025,R-2025-01,M-2025-005,Tottenham Hotspur,3:0,FC Burnley,16/08/2025,Michael Oliver,61.077
...,...,...,...,...,...,...,...,...,...
375,PL-2025,R-2025-38,M-2025-376,Manchester City FC,1:2,Aston Villa FC,24/05/2026,Andrew Madley,60.332
376,PL-2025,R-2025-38,M-2025-377,Nottingham Forest,1:1,AFC Bournemouth,24/05/2026,Craig Pawson,30.741
377,PL-2025,R-2025-38,M-2025-378,AFC Sunderland,2:1,Chelsea FC,24/05/2026,Chris Kavanagh,
378,PL-2025,R-2025-38,M-2025-379,Tottenham Hotspur,1:0,FC Everton,24/05/2026,Michael Oliver,61.812
